## 1. Reconciliación de filas y 2. Datos rescatados

Observaciones:
- 2256 filas en bronze = 2257 filas del Excel menos el encabezado
- No hubo datos rescatados en ninguna fila `_rescued_data`

In [0]:
%sql

-- Conteo de filas totales y filas rescatadas

select 
    count(*) as total_filas,
    sum(case when _rescued_data is not null then 1 else 0 end) as filas_con_rescate 
from fabriperezchauca.bronze.ventas

## 3. Identificación de filas vacías

Observaciones:
- 4 filas sin ningún dato del negocio (Excluir en silver)


In [0]:
%sql

-- Conteo de filas completamente vacías (todas las columnas del negocio en null)

select
    count(*) AS total_filas,
    count_if(
        coalesce(fecha, cantidad, producto, talla, precio_dado, ajuste_precio,
                 precio_original, precio_total, forma_pago, regateo, categoria,
                 observaciones) IS NULL
    ) AS filas_vacias
from fabriperezchauca.bronze.ventas

### Grano de la tabla de ventas

Observaciones:
- En silver, una fila = un producto vendido (con su cantidad) en una fecha
- El archivo no tiene número de venta ni de comprobante. No se puede saber qué filas forman una misma venta
- No se agrupan filas por fecha ni por forma de pago para reconstruir ventas. Bronze conserva el grano del archivo (una fila = una fila del Excel)
- Recomendar que registren un número de venta por transacción

In [0]:
%sql 
create or replace temp view ventas_no_vacias as 
select *
from fabriperezchauca.bronze.ventas
where coalesce(fecha, cantidad, producto, talla, precio_dado, ajuste_precio,
               precio_original, precio_total, forma_pago, regateo, categoria,
               observaciones) IS NOT NULL

## 4. Nulos por columnas + análisis de resultados

Observaciones:
- De 2252 filas: talla 2204 (97,9 %), observaciones 2199 (97,6 %), forma_pago 18, regateo 15, producto 12, cantidad 10, categoria 10, precio_original 9, precio_dado 8, precio_total 6, fecha 2, ajuste_precio 2
- talla y observaciones son opcionales en la práctica. Las demás son obligatorias: definir tratamiento en el EDA por KPI

In [0]:
%sql

-- Conteo de nulos por columna, excluyendo las filas completamente vacías

SELECT
    count(*)                            AS total_filas,
    count(*) - count(fecha)             AS fecha_null,
    count(*) - count(cantidad)          AS cantidad_null,
    count(*) - count(producto)          AS producto_null,
    count(*) - count(talla)             AS talla_null,
    count(*) - count(precio_dado)       AS precio_dado_null,
    count(*) - count(ajuste_precio)     AS ajuste_precio_null,
    count(*) - count(precio_original)   AS precio_original_null,
    count(*) - count(precio_total)      AS precio_total_null,
    count(*) - count(forma_pago)        AS forma_pago_null,
    count(*) - count(regateo)           AS regateo_null,
    count(*) - count(categoria)         AS categoria_null,
    count(*) - count(observaciones)     AS observaciones_null
FROM ventas_no_vacias

### ``ajuste_precio``

Observaciones:
- Faltan ambos precios → ``$-`` (0 falso)
- Falta uno → ``#¡VALOR!``
- Falta todo → ``null`` o ``$-``
- 13 filas con ``precio_original`` o ``precio_dado`` = null
- **Recalcular el ajuste en silver**

`ajuste_precio` no se toma del archivo: se recalcula en silver como `cantidad × (precio_original − precio_dado)`. Si alguna de las tres es null, el ajuste es null.

El ajuste queda null y la fila no cuenta para el KPI 3

In [0]:
%sql
select cantidad, precio_original, precio_dado, ajuste_precio
from fabriperezchauca.bronze.ventas
where precio_original is null or precio_dado is null

### ``precio_total``

Observaciones:
- ``precio_total`` cuando falta ``precio_dado`` = null
- 10 filas con ``precio_dado`` = null

In [0]:
%sql
select cantidad, precio_original, precio_dado, ajuste_precio, precio_total
from fabriperezchauca.bronze.ventas
where precio_total is null

### ``fecha``

Observaciones:
- `fecha` no es null
-  Excluir la fila: todos los KPIs se calculan por mes

In [0]:
%sql

-- Conteo de nulos en "fecha", relacionado a V-05

select *
from fabriperezchauca.bronze.ventas
where fecha is null

## 5. Cardinalidad

Estandarizar las cuatro columnas en silver. Revisar los valores de cada una para definir el mapeo

In [0]:
%sql
select 
    count(distinct(producto)) as total_productos,
    count(distinct(categoria)) as total_categorias,
    count(distinct(forma_pago)) as total_formas_pago,
    count(distinct(regateo)) as total_regateos
from fabriperezchauca.bronze.ventas

### ``regateo``

Observaciones:
- Después de trim y upper, el valor es `SI`, `NO` o null
- Valor fuera de la lista: marcar para revisión

In [0]:
%sql
select 
    concat('[' , regateo , ']') as valor_visible,
    count(*) as filas
from ventas_no_vacias
group by regateo
order by filas desc

### ``forma_pago``

Observaciones:
- Después de trim, minúsculas y mapeo, el valor es uno de: `de una`, `efectivo`, `transferencia`, `tarjeta de credito`, o null
- Valor fuera de la lista: marcar para revisión

In [0]:
%sql
select 
    concat('[' , forma_pago , ']') as valor_visible,
    count(*) as filas
from ventas_no_vacias
group by forma_pago
order by filas desc

### ``categoria``

Observaciones: 
- Después de trim, minúsculas y mapeo, el valor es uno de: `accesorios`, `uniformes`, `equipo tactico`, `equipamiento campo`, `calzado`, `otros`, o null
- Valor fuera de la lista: marcar para revisió

In [0]:
%sql
select 
    concat('[' , categoria , ']') as valor_visible,
    count(*) as filas
from ventas_no_vacias
group by categoria
order by filas desc

In [0]:
%sql

-- Listar "otros" para saber que tipos de productos hay

select 
    producto,
    categoria
from ventas_no_vacias
where lower(trim(categoria)) = 'otros'

### `categoria` = otros + `producto` = arreglos local

Observaciones: 
- Cuando `categoria` = otros + `producto` = arreglos local, se determina que es un **egreso**, tiene valor positivo y hay que excluirlos en silver
- Si `producto` = arreglo, es diferente del caso anterior, ese sí es un producto válido (servicio del negocio) y no debe confundirse
- 18 filas detectadas como egresos

In [0]:
%sql

-- Distinguir egresos filtrando "arreglos local"

select *
from ventas_no_vacias
where lower(trim(categoria)) = 'otros' and lower(trim(producto)) = 'arreglos local'

### `producto`

Normalización de productos.
Observaciones:
- El KPI 2 muestra el top de productos. Lo que importa es que los productos que llegan al top estén bien unificados. 
- Un producto que se vendió una sola vez, mal escrito, no cambia el ranking.
- 708 valores distintos. Con trim + minúsculas bajan a 595: el resto son variantes de escritura o productos distintos
- No se mapean los 595: solo los productos con más filas, que son los que forman el top del KPI 2

In [0]:
%sql

-- Normalización de productos

select count(distinct lower(trim(producto))) as productos_normalizados
from ventas_no_vacias

%md
### `producto`: top 40

Observaciones:
- Se excluyen `null`, `arreglos local` (egreso) y `valor de ventas` (no es un producto): no entran al KPI 2
- Sin variantes de escritura en el top 40 después de trim + minúsculas
- **En silver: solo trim + minúsculas, sin mapeo manual**
- Limitación: un producto con varias variantes de pocas filas cada una podría quedar fuera del top

In [0]:
%sql
select 
    lower(trim(producto)) as producto_normalizado,
    count(*) as filas
from ventas_no_vacias
where producto is not null and lower(trim(producto)) not in ('valor de ventas', 'arreglos local')
group by lower(trim(producto))
order by filas desc
limit 40

## 6. Valores que no se pueden convertir a número

Observaciones:
- Cuenta los valores no null que no se pueden convertir a número con `try_cast`

In [0]:
%sql
select 
    count_if(cantidad is not null and try_cast(cantidad as double) is null) as cantidad_invalida,
    count_if(precio_original is not null and try_cast(precio_original as double) is null) as precio_original_invalido,
    count_if(precio_dado is not null and try_cast(precio_dado as double) is null) as precio_dado_invalido,
    count_if(precio_total is not null and try_cast(precio_total as double) is null) as precio_total_invalido
from ventas_no_vacias

In [0]:
%sql
select 
    concat('[', precio_dado, ']') as valor_visible
from ventas_no_vacias
where precio_dado is not null and try_cast(precio_dado as double) is null
limit 20

### Precios con formato de moneda

Observaciones:
- Los precios vienen como texto con formato: ` $10.00 ` (espacios, símbolo `$`, punto decimal)
- Algunos tienen espacios invisibles (non-breaking space) que `trim()` no quita
- **En silver: `regexp_replace(columna, '[^0-9.-]', '')` y luego convertir a número**

In [0]:
%sql
select
    count_if(precio_original is not null and try_cast(regexp_replace(precio_original, '[^0-9.-]', '') as double) is null) as precio_original_invalido,
    count_if(precio_dado     is not null and try_cast(regexp_replace(precio_dado,     '[^0-9.-]', '') as double) is null) as precio_dado_invalido,
    count_if(precio_total    is not null and try_cast(regexp_replace(precio_total,    '[^0-9.-]', '') as double) is null) as precio_total_invalido
from ventas_no_vacias

### Valores que quedan null al limpiar precios

Observaciones:
- `precio_original` y `precio_dado`: 0 valores inválidos después de la limpieza
- `precio_total`: 3 inválidos → `$-` (2, cero falso) y `#¡VALOR!` (1, error de fórmula)
- Los dos casos quedan null al convertir, que es el resultado correcto: no son montos reales

In [0]:
%sql
select 
    concat('[', precio_total, ']') as valor_visible
from ventas_no_vacias
where precio_total is not null and try_cast(regexp_replace(precio_total,    '[^0-9.-]', '') as double) is null
limit 20

In [0]:
%sql
select  
    concat('[', fecha, ']') as valor_visible
from ventas_no_vacias
where fecha is not null
limit 20

In [0]:
%sql
select 
    count_if(fecha is not null and try_to_date(fecha, 'dd/MM/yyyy') is null) as fecha_invalida
from ventas_no_vacias

In [0]:
%sql
select 
    concat('[', fecha, ']') as valor_visible
from ventas_no_vacias
where fecha is not null and try_to_date(fecha, 'dd/MM/yyyy') is null

### `fecha`

Observaciones:
- Formato `dd/MM/yyyy`
- 20 fechas con guion en lugar de barra antes del año (`11/09-2026`): error de digitación
- **En silver: `try_to_date(replace(fecha, '-', '/'), 'dd/MM/yyyy')`**

In [0]:
%sql
select 
    count_if(fecha is not null and try_to_date(replace(fecha, '-', '/'), 'dd/MM/yyyy') is null) as fecha_invalida
from ventas_no_vacias

## 7. Duplicados

Observaciones:
- 64 grupos de filas idénticas, 76 filas extra
- El caso más repetido (5 veces) es bordado de nombres: barato y común, tiene sentido como ventas reales
- **No se eliminan duplicados en silver**: sin número de venta no se puede distinguir una venta repetida de una doble digitación

In [0]:
%sql
select 
    fecha,
    cantidad,
    producto,
    talla,
    precio_dado,
    precio_original,
    precio_total,
    forma_pago,
    regateo,
    categoria,
    observaciones,
    count(*) as veces
from ventas_no_vacias
group by all
having count(*) > 1
order by veces desc

In [0]:
%sql

with repeticiones as (
    select 
        fecha,
        cantidad,
        producto,
        talla,
        precio_dado,
        precio_original,
        precio_total,
        forma_pago,
        regateo,
        categoria,
        observaciones,
        count(*) as veces
    from ventas_no_vacias
    group by all
    having count(*) > 1
)

select 
    count(*) as grupos_repetidos,
    sum(veces) - count(*) as filas_extra
from repeticiones


## 8. Rangos y outliers

Observaciones:
- `cantidad` de 1 a 80 y precios de $0.25 a $800: los máximos son ventas al por mayor y contratos, **se mantienen**
- 25 fechas con el año mal digitado (`0202` en lugar de `2026`).
- Rango final de fechas después de la corrección: 2026-06-01 a 2026-11-09
- **En silver: si el año no es 2026, se reemplaza por 2026.** El día puede quedar inexacto, pero los KPIs son mensuales y no se ven afectados

In [0]:
%sql

with convertido as (
    select 
        try_to_date(replace(fecha, '-', '/'), 'dd/MM/yyyy') as fecha,
        try_cast(cantidad as double),
        try_cast(regexp_replace(precio_original, '[^0-9.-]', '') as double) as precio_original,
        try_cast(regexp_replace(precio_dado, '[^0-9.-]', '') as double) as precio_dado,
        try_cast(regexp_replace(precio_total, '[^0-9.-]', '') as double) as precio_total
    from ventas_no_vacias
)

select 
    min(fecha), max(fecha),
    min(cantidad), max(cantidad),
    min(precio_original), max(precio_original),
    min(precio_dado), max(precio_dado),
    min(precio_total), max(precio_total)
from convertido

In [0]:
%sql
select 
    fecha as fecha_original,
    try_to_date(replace(fecha, '-', '/'), 'dd/MM/yyyy') as fecha_convertida
from ventas_no_vacias
where year(try_to_date(replace(fecha, '-', '/'), 'dd/MM/yyyy')) <> 2026

In [0]:
%sql

with fecha_formateada as (
    select try_to_date(replace(fecha, '-', '/'), 'dd/MM/yyyy') as f
    from ventas_no_vacias
)

select 
    min(make_date(2026, month(f), day(f))) as fecha_min,
    max(make_date(2026, month(f), day(f))) as fecha_max
from fecha_formateada

## 9. Análisis temporal

Observaciones:
- Junio, julio y agosto: ~27 días con ventas cada uno (completos)
- Septiembre: 15 días. Noviembre: 5 días (hasta el 09/11). Parciales
- **Octubre: sin ninguna fila**
- Confirmado con el negocio: los registros de octubre se perdieron. Septiembre y noviembre también tienen días sin registrar. La app se empezó a usar el 19/11, así que esas ventas no están en otro sistema
- **Para gold: octubre se muestra como "sin datos", nunca como $0.** Septiembre y noviembre se marcan como parciales. Solo junio, julio y agosto son comparables entre sí
- Si el negocio encuentra los registros de octubre, se agregan al archivo en landing y se vuelve a ejecutar el pipeline


In [0]:
%sql
with fechas as (
    select try_to_date(replace(fecha, '-', '/'), 'dd/MM/yyyy') AS f
    from ventas_no_vacias
),
anio_valido as (
    select make_date(2026, month(f), day(f)) as fecha
    from fechas
    where f is not null
)

select 
    month(fecha) as mes,
    count(*) as filas,
    count(distinct fecha) as dias_con_ventas
from anio_valido
group by month(fecha)
order by mes